# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tajmomin/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# ML-03 — Frame Your Lane as an ML Task

## 1. My lane as an ML task (type)

**Task Type:** Supervised Binary Classification coupled with Probability-Based Priority Ranking.

**Why this formulation:**
A pure regression or continuous decay model introduces noise across low-traffic pages where absolute delta fluctuations are volatile. Instead, framing the task as predicting the probability of content decline ($P(\text{decline} = 1 \mid X)$) gives a calibrated risk score. We then rank candidates by combining predicted decline risk with search exposure (demand weight), creating an actionable queue for capacity-constrained editorial workflows.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Search for existing CSV in the local runtime
filename = "content_refresh_anonymized.csv"
found = list(Path("/content").rglob(filename))

if found:
    data_path = found[0]
else:
    # 2. If repo not cloned yet, clone it
    if not Path("/content/flyrank").exists():
        !git clone https://github.com/tajmomin/flyrank.git /content/flyrank

    # 3. Check again after clone
    found = list(Path("/content/flyrank").rglob(filename))
    if found:
        data_path = found[0]
    else:
        # 4. Fallback: Download the starter CSV directly if missing from git
        os.makedirs("/content/flyrank/data/raw", exist_ok=True)
        data_path = Path("/content/flyrank/data/raw/content_refresh_anonymized.csv")
        !wget -q -O {data_path} https://raw.githubusercontent.com/tajmomin/flyrank/main/data/raw/content_refresh_anonymized.csv

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path)

# Validate binary target representation
df["target_declining"] = (df["trend_direction"] == "down").astype(int)
class_distribution = df["target_declining"].value_counts(normalize=True)

print(f"\nTarget distribution (0: Stable/Growing, 1: Declining):\n{class_distribution}")
print(f"\nTotal candidate instances for ranking: {len(df):,}")

Cloning into '/content/flyrank'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 38), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.87 MiB | 13.29 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Loading data from: /content/flyrank/data/raw/content_refresh_anonymized.csv

Target distribution (0: Stable/Growing, 1: Declining):
target_declining
1    0.542067
0    0.457933
Name: proportion, dtype: float64

Total candidate instances for ranking: 30,000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

* **What we predict:** The binary state $Y \in \{0, 1\}$, indicating whether a content asset is experiencing significant downward trajectory (`trend_direction == 'down'`).
* **Source of the label:** In this starter slice, the label is a derived proxy calculated from observable historical window metrics rather than a forward-looking test period. In the full warehouse extension, this label transitions to a true future-window observed outcome (e.g., prior 90 days feature window predicting next 30 days traffic drop) to eliminate contemporaneous signal leakage.
* **Leakage Safeguard:** Derived flags computed directly from the current target window (such as product-level decision fields or raw trend percentages) are strictly excluded from the feature matrix $X$.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify proxy label balance across demand tiers without using forbidden decision flags
high_demand_threshold = 500
tier_counts = (
    df.groupby(df["impressions_90d"] >= high_demand_threshold)[
        "target_declining"
    ]
    .mean()
    .rename(index={False: "Low/Med Demand (<500)", True: "High Demand (>=500)"})
)

print(
    f"Observed decline rate across demand segments:\n{tier_counts.round(4)}"
)

Observed decline rate across demand segments:
impressions_90d
Low/Med Demand (<500)    0.4747
High Demand (>=500)      0.5955
Name: target_declining, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*
## 3. Success metric

* **Primary Defensible Metric:** **Precision@50** (and Average Precision / PR-AUC).
* **Why this metric matters:** An editorial team has finite review capacity (e.g., 50 pages per review cycle). Generic Accuracy or ROC-AUC can be inflated by correctly classifying thousands of low-impact, non-declining tail pages. Precision@50 measures the exact fraction of actionable, true positives surfaced in the top 50 recommendations of the queue.
* **What number means 'good':**
  * Fixed baseline heuristic achieves **Precision@50 = 0.240** (12 out of 50 correct).
  * A satisfactory ML model should achieve **Precision@50 $\ge$ 0.700** (35+ out of 50 correct), representing an approximate 3x improvement in reviewer efficiency.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import average_precision_score, precision_score

# Demonstrate the capacity-constrained top-K evaluation metric logic
# Simulating a mock score to demonstrate Precision@K computation
np.random.seed(42)
mock_heuristic_rank = df["impressions_90d"] / (
    df["content_age_days"] + 1
)  # Simple naive heuristic
top_50_idx = mock_heuristic_rank.sort_values(ascending=False).head(50).index
simulated_p50 = precision_score(
    df.loc[top_50_idx, "target_declining"], np.ones(50, dtype=int)
)

print(f"Baseline Naive Heuristic Precision@50 on starter slice: {simulated_p50:.3f}")
print("Target benchmark to beat: Precision@50 >= 0.700 (Random Forest starter benchmark = 0.740)")

Baseline Naive Heuristic Precision@50 on starter slice: 0.340
Target benchmark to beat: Precision@50 >= 0.700 (Random Forest starter benchmark = 0.740)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

* **Granularity (Grain):** Exactly **one row per pseudonymized content item** (`content_id`), aggregated across its historical performance window.
* **Key Identifiers:** `content_id` (entity primary key) and `client_id` (grouping key for client-holdout cross-validation).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display the exact unit of analysis dataframe structure
sample_slice = df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "content_age_days",
        "ctr",
        "target_declining",
    ]
].head(5)

print("DataFrame Grain: 1 row = 1 unique content asset (content_id)")
print(f"Is content_id unique across rows? {df['content_id'].is_unique}")
print(f"Total rows (N): {len(df):,}")
sample_slice

DataFrame Grain: 1 row = 1 unique content asset (content_id)
Is content_id unique across rows? True
Total rows (N): 30,000


,content_id,client_id,impressions_90d,clicks_90d,avg_position,content_age_days,ctr,target_declining
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,187,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,445,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,141,0.09,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,463,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,263,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

* **Non-Linear Interactions:** Fixed IF-statements fail to balance multi-dimensional trade-offs (e.g., a page with high impressions at position 18 with high CTR vs. a page at position 3 with decaying CTR and high age).
* **Continuous Trade-off Surfaces:** Static thresholds (like `content_age_days >= 180 AND impressions >= 500`) produce hard boundary artifacts, treating 179 days as completely risk-free and 180 days as urgent. ML maps these into a smooth probability space.
* **Heterogeneous Client Baselines:** Search dynamics differ across verticals; a learned model captures complex feature correlations across engagement, volume, and positioning that cannot be captured by hand-crafted heuristics.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Illustrate boundary failure of static rule thresholds
# Compare pages right on either side of an arbitrary 180-day cutoff
edge_case_179 = df[
    (df["content_age_days"] >= 170)
    & (df["content_age_days"] < 180)
    & (df["impressions_90d"] >= 500)
]
edge_case_180 = df[
    (df["content_age_days"] >= 180)
    & (df["content_age_days"] < 190)
    & (df["impressions_90d"] >= 500)
]

rate_179 = edge_case_179["target_declining"].mean()
rate_180 = edge_case_180["target_declining"].mean()

print(
    f"Decline rate for age 170-179 days (excluded by naive rule): {rate_179:.2%}"
)
print(f"Decline rate for age 180-189 days (captured by naive rule): {rate_180:.2%}")
print(
    f"Empirical difference: {abs(rate_180 - rate_179):.2%} — Static threshold creates an artificial discontinuity."
)

Decline rate for age 170-179 days (excluded by naive rule): 46.75%
Decline rate for age 180-189 days (captured by naive rule): 81.13%
Empirical difference: 34.38% — Static threshold creates an artificial discontinuity.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.